# 🧠 Brain Tumor MRI Classification with Grad-CAM Explainability

<div style='background:#1a1a2e; padding:20px; border-radius:12px; border-left:5px solid #e94560; color:#eee;'>
<b>⚕️ Medical Disclaimer:</b> This project is strictly for <b>educational and research purposes</b>. It is NOT a medical diagnostic tool. All predictions must be interpreted by qualified medical professionals. Never use this system for actual clinical decision-making.
</div>

---

## 📋 Project Overview

This end-to-end deep learning project classifies brain MRI scans into **4 categories** using state-of-the-art transfer learning models and visualizes predictions with **Grad-CAM heatmaps** — making the model's reasoning interpretable to clinicians.

### 🎯 Classes
| Class | Description | Prevalence |
|-------|-------------|------------|
| **Glioma** | Tumors arising from glial cells; most common primary brain tumor | ~30% of cases |
| **Meningioma** | Tumors from meninges (brain lining); usually benign | ~36% of cases |
| **Pituitary** | Tumors of the pituitary gland; often hormone-secreting | ~15% of cases |
| **No Tumor** | Normal brain MRI scan | Control group |

### 🧪 Deep Learning Pipeline
```
Raw MRI Images (DICOM/JPG)
       │
       ▼
Medical Preprocessing (CLAHE, skull-strip simulation, normalization)
       │
       ▼
Data Augmentation (rotation, zoom, brightness, elastic deformation)
       │
       ├──────────────────────────────────────────────┐
       │                                              │
  Baseline CNN                     Transfer Learning Models
  (built from scratch)         ResNet50 / EfficientNetB0 / MobileNetV2
       │                                              │
       └──────────────────────────────────────────────┘
                               │
                               ▼
              Model Evaluation (Accuracy, AUC, F1)
                               │
                               ▼
               Grad-CAM Explainability Heatmaps
                               │
                               ▼
                  Clinical Uncertainty Estimation
```

### 🔬 Resume-Worthy Features
- ✅ Multi-model benchmark (4 architectures)
- ✅ Grad-CAM + Guided Backpropagation visualization
- ✅ Medical preprocessing (CLAHE, segmentation)
- ✅ Class-weighted training for imbalanced data
- ✅ Monte Carlo Dropout uncertainty estimation
- ✅ ROC curves + AUC per class
- ✅ t-SNE feature space visualization
- ✅ Learning curves & training diagnostics
- ✅ Clinical threshold calibration
- ✅ Ensemble model fusion

## 📦 Section 1: Installation & Environment Setup

We install all required packages. Key libraries:
- **TensorFlow/Keras**: Deep learning framework for model building and training
- **OpenCV**: Computer vision for medical image preprocessing
- **scikit-learn**: Evaluation metrics, train/test splits
- **albumentations**: Medical-grade image augmentation
- **matplotlib/seaborn**: Visualization and analysis plots
- **grad-cam**: Gradient-weighted Class Activation Mapping

In [ ]:
# Install all dependencies
!pip install tensorflow opencv-python scikit-learn matplotlib seaborn numpy pandas \
             albumentations tqdm Pillow scipy kaggle --quiet

# Optional: Install tf-keras-vis for advanced Grad-CAM
!pip install tf-keras-vis --quiet

print('✅ All packages installed successfully!')

## 🔧 Section 2: Imports & Global Configuration

We organize imports by domain and set all experiment hyperparameters in a single `CONFIG` dictionary for reproducibility.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import os
import json
import warnings
import random
import time
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ── Numerical & Data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Computer Vision ───────────────────────────────────────────────────────────
import cv2
from PIL import Image

# ── Deep Learning ─────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, optimizers, callbacks
from tensorflow.keras.applications import ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

# ── Machine Learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, 
    roc_auc_score, roc_curve, auc,
    f1_score, precision_recall_curve
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.manifold import TSNE
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import ndimage

# ── Utilities ─────────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Global Configuration ──────────────────────────────────────────────────────
CONFIG = {
    # Dataset
    'data_dir': './brain_tumor_dataset',  # Update after Kaggle download
    'classes': ['glioma', 'meningioma', 'notumor', 'pituitary'],
    'class_display': ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary'],
    
    # Image settings
    'img_size': 224,        # Standard for ImageNet pre-trained models
    'channels': 3,          # RGB (MRI grayscale → converted to 3-channel)
    
    # Training
    'batch_size': 32,
    'epochs_baseline': 20,
    'epochs_transfer': 15,  # Fine-tuning epochs
    'epochs_finetune': 10,  # Full fine-tuning
    'learning_rate': 1e-4,
    'finetune_lr': 1e-5,
    
    # Splits
    'train_ratio': 0.70,
    'val_ratio': 0.15,
    'test_ratio': 0.15,
    
    # MC Dropout (uncertainty estimation)
    'mc_samples': 50,       # Number of stochastic forward passes
    'dropout_rate': 0.5,
    
    # Grad-CAM
    'gradcam_layer_resnet': 'conv5_block3_out',
    'gradcam_layer_efficientnet': 'top_conv',
    'gradcam_layer_mobilenet': 'Conv_1',
    
    # Colors for visualization
    'class_colors': ['#e74c3c', '#3498db', '#2ecc71', '#f39c12'],
}

# Custom colormap for Grad-CAM (jet-like but medical)
GRADCAM_CMAP = plt.cm.jet

# Plotting style
plt.rcParams.update({
    'figure.facecolor': '#0a0a0f',
    'axes.facecolor': '#12121a',
    'axes.edgecolor': '#2a2a3e',
    'text.color': '#e0e0e0',
    'axes.labelcolor': '#e0e0e0',
    'xtick.color': '#9999bb',
    'ytick.color': '#9999bb',
    'grid.color': '#2a2a3e',
    'figure.dpi': 120,
})

print(f'✅ TensorFlow version: {tf.__version__}')
print(f'🖥️  GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')
print(f'📊 Classes: {CONFIG["class_display"]}')
print(f'🖼️  Image size: {CONFIG["img_size"]}×{CONFIG["img_size"]}×{CONFIG["channels"]}')

## 📥 Section 3: Dataset Download from Kaggle

### Dataset: Brain Tumor MRI Dataset
- **Source**: [Kaggle - Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)
- **Size**: ~7,200 MRI images
- **Format**: JPEG images organized in class folders
- **Resolution**: Variable (we'll standardize to 224×224)

### How to Download
1. Create Kaggle account at kaggle.com
2. Go to Account → Create New API Token → download `kaggle.json`
3. Place `kaggle.json` in `~/.kaggle/` directory
4. Run the cell below

In [ ]:
import os
from pathlib import Path

def download_dataset():
    """Download Brain Tumor MRI dataset from Kaggle."""
    
    # Check if kaggle credentials exist
    kaggle_json = Path.home() / '.kaggle' / 'kaggle.json'
    
    if not kaggle_json.exists():
        print('⚠️  kaggle.json not found!')
        print('   Steps to fix:')
        print('   1. Go to https://www.kaggle.com/account')
        print('   2. Click "Create New API Token"')
        print('   3. Move downloaded kaggle.json to ~/.kaggle/')
        print('   4. Run: chmod 600 ~/.kaggle/kaggle.json')
        return False
    
    data_dir = Path(CONFIG['data_dir'])
    if data_dir.exists() and any(data_dir.iterdir()):
        print(f'✅ Dataset already exists at {data_dir}')
        return True
    
    print('📥 Downloading Brain Tumor MRI Dataset from Kaggle...')
    os.system('kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset --unzip -p ./brain_tumor_dataset')
    print('✅ Download complete!')
    return True


def create_synthetic_dataset(n_per_class: int = 50):
    """
    Create a synthetic dataset for demonstration when Kaggle is not available.
    
    Generates realistic grayscale MRI-like images with class-specific patterns:
    - Glioma: Large irregular high-intensity region
    - Meningioma: Peripheral bright ring
    - Pituitary: Small central bright spot
    - No Tumor: Uniform brain tissue pattern
    """
    print(f'🔬 Creating synthetic MRI dataset ({n_per_class} images per class)...')
    base_path = Path(CONFIG['data_dir'])
    
    for split in ['Training', 'Testing']:
        n = n_per_class if split == 'Training' else n_per_class // 4
        for cls in CONFIG['classes']:
            cls_path = base_path / split / cls
            cls_path.mkdir(parents=True, exist_ok=True)
            
            for i in range(n):
                img = generate_mri_image(cls, i)
                img_path = cls_path / f'{cls}_{i:04d}.jpg'
                cv2.imwrite(str(img_path), img)
    
    print('✅ Synthetic dataset created!')


def generate_mri_image(tumor_class: str, seed: int = 0) -> np.ndarray:
    """
    Generate a synthetic brain MRI image with class-specific tumor patterns.
    Uses Gaussian blobs, circular masks, and realistic noise to simulate MRI texture.
    """
    np.random.seed(seed * 100 + hash(tumor_class) % 1000)
    size = 256
    
    # 1. Base brain tissue (elliptical with noise)
    img = np.zeros((size, size), dtype=np.float32)
    Y, X = np.ogrid[:size, :size]
    cx, cy = size // 2 + np.random.randint(-10, 10), size // 2 + np.random.randint(-10, 10)
    rx, ry = size // 2 - 20 + np.random.randint(-5, 5), size // 2 - 15 + np.random.randint(-5, 5)
    brain_mask = ((X - cx) / rx) ** 2 + ((Y - cy) / ry) ** 2 <= 1
    img[brain_mask] = 0.4 + np.random.normal(0, 0.05, brain_mask.sum()).clip(0, 1)
    
    # 2. Add tissue layers (gray/white matter boundaries)
    inner_rx, inner_ry = rx - 15, ry - 15
    inner_mask = ((X - cx) / inner_rx) ** 2 + ((Y - cy) / inner_ry) ** 2 <= 1
    img[inner_mask] = 0.55 + np.random.normal(0, 0.04, inner_mask.sum()).clip(0, 1)
    
    # 3. Class-specific tumor patterns
    if tumor_class == 'glioma':
        # Large irregular high-intensity blob (infiltrative)
        tx = cx + np.random.randint(-40, 40)
        ty = cy + np.random.randint(-30, 30)
        for _ in range(4):  # Multi-focal
            r = np.random.randint(20, 45)
            x_off = np.random.randint(-15, 15)
            y_off = np.random.randint(-15, 15)
            tumor_mask = (X - (tx + x_off)) ** 2 + (Y - (ty + y_off)) ** 2 <= r ** 2
            tumor_mask &= brain_mask
            img[tumor_mask] = 0.85 + np.random.normal(0, 0.07, tumor_mask.sum()).clip(0, 1)
        # Necrotic core (dark center)
        core_mask = (X - tx) ** 2 + (Y - ty) ** 2 <= 12 ** 2
        img[core_mask & brain_mask] = 0.15
        
    elif tumor_class == 'meningioma':
        # Peripheral bright ring attached to skull base
        angle = np.random.uniform(0, 2 * np.pi)
        mx = int(cx + (rx - 25) * np.cos(angle))
        my = int(cy + (ry - 25) * np.sin(angle))
        r = np.random.randint(18, 35)
        tumor_mask = (X - mx) ** 2 + (Y - my) ** 2 <= r ** 2
        img[tumor_mask] = 0.80 + np.random.normal(0, 0.05, tumor_mask.sum()).clip(0, 1)
        # Dural tail (characteristic feature)
        for t in np.linspace(0, 1, 20):
            dx = int(mx + t * 15 * np.cos(angle + np.pi))
            dy = int(my + t * 15 * np.sin(angle + np.pi))
            tail_mask = (X - dx) ** 2 + (Y - dy) ** 2 <= (3 - t * 2) ** 2
            if 0 <= dx < size and 0 <= dy < size:
                img[tail_mask] = 0.78
                
    elif tumor_class == 'pituitary':
        # Small central bright spot (sella turcica region)
        px = cx + np.random.randint(-8, 8)
        py = cy + 15 + np.random.randint(-5, 5)  # Slightly inferior
        r = np.random.randint(10, 20)
        tumor_mask = (X - px) ** 2 + (Y - py) ** 2 <= r ** 2
        img[tumor_mask & brain_mask] = 0.90 + np.random.normal(0, 0.04, tumor_mask.sum()).clip(0, 1)
        
    # else: notumor — no additional pattern
    
    # 4. Add realistic MRI noise (Rician noise model)
    noise = np.random.normal(0, 0.025, img.shape)
    img = np.clip(img + noise, 0, 1)
    
    # 5. Apply slight Gaussian smoothing (MRI PSF simulation)
    img = ndimage.gaussian_filter(img, sigma=1.2)
    
    # 6. Convert to uint8 RGB (3-channel for model compatibility)
    img_uint8 = (img * 255).astype(np.uint8)
    img_rgb = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)
    
    return img_rgb


# Run download or create synthetic
if not download_dataset():
    print('\n📋 Using synthetic dataset for demonstration...')
    create_synthetic_dataset(n_per_class=200)

## 🏥 Section 4: Medical Image Preprocessing Pipeline

### Why Medical Preprocessing is Different

Unlike natural images, MRI scans require specialized preprocessing:

1. **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: Enhances local contrast without amplifying noise — critical for revealing subtle tumor boundaries
2. **Brain Extraction (Skull Stripping)**: Removes non-brain tissue to focus the model on relevant anatomy
3. **Normalization**: Z-score normalization per-image (not global) since MRI intensity is arbitrary across scanners
4. **Intensity Clipping**: Remove extreme outliers (motion artifacts, scanner artifacts)

### CLAHE Explained
Standard histogram equalization enhances contrast globally but can wash out local tumor structures. CLAHE divides the image into tiles and applies equalization locally, preserving the appearance of heterogeneous tumor tissue.

In [ ]:
def apply_clahe(image: np.ndarray, clip_limit: float = 2.0, tile_size: int = 8) -> np.ndarray:
    """
    Apply Contrast Limited Adaptive Histogram Equalization (CLAHE).
    
    CLAHE works by:
    1. Dividing image into non-overlapping tiles (tile_size × tile_size)
    2. Computing histogram for each tile
    3. Clipping histogram at clip_limit to prevent over-amplification
    4. Redistributing clipped counts uniformly
    5. Interpolating between tiles for smooth output
    
    Args:
        clip_limit: Threshold for contrast limiting (higher = more contrast)
        tile_size: Grid size for local histogram computation
    """
    if len(image.shape) == 3 and image.shape[2] == 3:
        # Convert to LAB color space → apply CLAHE only to L channel
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        l_channel, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
        cl = clahe.apply(l_channel)
        enhanced = cv2.merge([cl, a, b])
        return cv2.cvtColor(enhanced, cv2.COLOR_LAB2BGR)
    else:
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(tile_size, tile_size))
        return clahe.apply(image)


def skull_strip_simulation(image: np.ndarray) -> tuple:
    """
    Simulate brain extraction by isolating the largest connected bright region.
    
    Real skull stripping (FSL BET, FreeSurfer) uses complex atlas-based methods.
    This simplified version:
    1. Threshold at Otsu's method
    2. Find connected components
    3. Keep largest component (brain parenchyma)
    4. Apply as mask to original image
    
    Returns:
        masked_image: Brain-only image
        mask: Binary brain mask
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if len(image.shape) == 3 else image
    
    # Otsu's thresholding
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # Morphological closing to fill holes
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    closed = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    
    # Find connected components
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(closed)
    
    if num_labels <= 1:
        return image, np.ones_like(gray, dtype=np.uint8) * 255
    
    # Largest component (excluding background at index 0)
    largest = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    mask = (labels == largest).astype(np.uint8) * 255
    
    # Apply mask
    masked = cv2.bitwise_and(image, image, mask=mask)
    return masked, mask


def preprocess_mri(image: np.ndarray, target_size: int = 224, 
                   apply_skull_strip: bool = True,
                   normalize: bool = True) -> np.ndarray:
    """
    Full medical image preprocessing pipeline.
    
    Pipeline steps:
    1. Resize to target size (224×224 for ImageNet-pretrained models)
    2. CLAHE enhancement for contrast normalization
    3. Optional skull stripping (removes non-brain tissue)
    4. Z-score normalization (zero mean, unit variance per image)
    5. Return float32 in [0, 1] range
    
    Args:
        image: Input BGR image (H×W×3)
        target_size: Output spatial dimensions
        apply_skull_strip: Whether to apply brain extraction
        normalize: Whether to apply z-score normalization
    """
    # Step 1: Resize
    img = cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_LANCZOS4)
    
    # Step 2: CLAHE enhancement
    img = apply_clahe(img, clip_limit=2.0, tile_size=8)
    
    # Step 3: Skull stripping
    if apply_skull_strip:
        img, _ = skull_strip_simulation(img)
    
    # Step 4: Convert to float
    img = img.astype(np.float32) / 255.0
    
    # Step 5: Per-image z-score normalization
    if normalize:
        mean = img.mean()
        std = img.std() + 1e-8  # Prevent division by zero
        img = (img - mean) / std
        # Clip to [-3, 3] std (removes extreme scanner artifacts)
        img = np.clip(img, -3.0, 3.0)
        # Rescale back to [0, 1] for model compatibility
        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    
    return img


# Demonstrate preprocessing steps visually
def visualize_preprocessing_pipeline(sample_image: np.ndarray):
    """Show each preprocessing step side by side."""
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    fig.patch.set_facecolor('#0a0a0f')
    
    steps = []
    titles = ['Original', 'CLAHE Enhanced', 'Skull Stripped', 'Normalized', 'Final (224×224)']
    
    # Original
    steps.append(cv2.cvtColor(sample_image, cv2.COLOR_BGR2RGB) if len(sample_image.shape)==3 else sample_image)
    
    # CLAHE
    clahe_img = apply_clahe(sample_image)
    steps.append(cv2.cvtColor(clahe_img, cv2.COLOR_BGR2RGB))
    
    # Skull strip
    stripped, mask = skull_strip_simulation(clahe_img)
    steps.append(cv2.cvtColor(stripped, cv2.COLOR_BGR2RGB))
    
    # Normalized
    norm_img = stripped.astype(np.float32) / 255.0
    mean, std = norm_img.mean(), norm_img.std() + 1e-8
    norm_img = np.clip((norm_img - mean) / std, -3, 3)
    norm_img = (norm_img - norm_img.min()) / (norm_img.max() - norm_img.min() + 1e-8)
    steps.append(norm_img)
    
    # Final resized
    final = preprocess_mri(sample_image, target_size=224)
    steps.append(final)
    
    for ax, img, title in zip(axes, steps, titles):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title, color='white', fontsize=11, fontweight='bold', pad=10)
        ax.axis('off')
    
    plt.suptitle('🏥 Medical Image Preprocessing Pipeline', 
                 color='white', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig('preprocessing_pipeline.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()


# Demo with a synthetic image
sample_img = generate_mri_image('glioma', seed=7)
visualize_preprocessing_pipeline(sample_img)
print('✅ Preprocessing pipeline demonstrated!')

## 📂 Section 5: Data Loading, Augmentation & Class Balance Analysis

### Medical Image Augmentation Strategy

For MRI images, augmentation must be **anatomically plausible**:
- ✅ **Rotation** (±20°): MRI scans can be taken at slight angles
- ✅ **Horizontal flip**: Brain has bilateral symmetry
- ✅ **Brightness/Contrast**: Simulates different scanner settings
- ✅ **Zoom** (±10%): Different patient sizes
- ✅ **Elastic deformation**: Simulates anatomical variations
- ❌ **Vertical flip**: NOT used — brain orientation is anatomically fixed
- ❌ **Color jitter**: NOT used — MRI grayscale values carry diagnostic meaning

### Class Imbalance Handling
Medical datasets are inherently imbalanced. We use:
1. **Class weights**: Penalize model more for misclassifying rare classes
2. **Stratified splitting**: Maintain class ratios in train/val/test
3. **Oversampling**: Augment minority classes more aggressively

In [ ]:
def load_dataset(data_dir: str, img_size: int = 224, max_per_class: int = None) -> tuple:
    """
    Load and preprocess MRI images from directory structure:
    data_dir/
        Training/
            glioma/
            meningioma/
            notumor/
            pituitary/
        Testing/
            ...
    
    Returns:
        X: Preprocessed images array (N, H, W, 3)
        y: Integer labels (N,)
        class_names: List of class names
    """
    data_path = Path(data_dir)
    class_names = CONFIG['classes']
    label_map = {cls: i for i, cls in enumerate(class_names)}
    
    images, labels, paths = [], [], []
    
    # Collect from Training and Testing splits
    for split in ['Training', 'Testing']:
        split_path = data_path / split
        if not split_path.exists():
            # Flat structure fallback
            split_path = data_path
        
        for cls in class_names:
            cls_path = split_path / cls
            if not cls_path.exists():
                continue
            
            img_files = list(cls_path.glob('*.jpg')) + list(cls_path.glob('*.jpeg')) + \
                        list(cls_path.glob('*.png'))
            
            if max_per_class:
                img_files = img_files[:max_per_class]
            
            print(f'  {split}/{cls}: {len(img_files)} images', end='\r')
            
            for img_path in img_files:
                img = cv2.imread(str(img_path))
                if img is None:
                    continue
                
                # Apply preprocessing pipeline
                processed = preprocess_mri(img, target_size=img_size)
                images.append(processed)
                labels.append(label_map[cls])
                paths.append(str(img_path))
    
    print()
    X = np.array(images, dtype=np.float32)
    y = np.array(labels, dtype=np.int32)
    
    return X, y, paths


def create_data_generators(X_train, y_train, X_val, y_val, X_test, y_test):
    """
    Create Keras data generators with augmentation.
    
    Training generator uses heavy augmentation.
    Validation/Test generators have NO augmentation (only normalization).
    
    Why: Augmentation during evaluation would give non-reproducible results
         and doesn't reflect real-world inference conditions.
    """
    # Training: with augmentation
    train_datagen = ImageDataGenerator(
        rotation_range=20,         # ±20° rotation (anatomically valid)
        width_shift_range=0.1,     # Horizontal translation
        height_shift_range=0.1,    # Vertical translation
        zoom_range=0.1,            # 10% zoom (different patient sizes)
        horizontal_flip=True,      # Brain bilateral symmetry
        brightness_range=[0.85, 1.15],  # Scanner variability
        fill_mode='nearest',       # Border fill mode for rotations
    )
    
    # Validation & Test: NO augmentation
    test_datagen = ImageDataGenerator()  # Identity (no augmentation)
    
    y_train_cat = to_categorical(y_train, len(CONFIG['classes']))
    y_val_cat = to_categorical(y_val, len(CONFIG['classes']))
    y_test_cat = to_categorical(y_test, len(CONFIG['classes']))
    
    train_gen = train_datagen.flow(X_train, y_train_cat, batch_size=CONFIG['batch_size'], seed=SEED)
    val_gen = test_datagen.flow(X_val, y_val_cat, batch_size=CONFIG['batch_size'], shuffle=False)
    test_gen = test_datagen.flow(X_test, y_test_cat, batch_size=CONFIG['batch_size'], shuffle=False)
    
    return train_gen, val_gen, test_gen


# Load dataset
print('📂 Loading dataset...')
X, y, img_paths = load_dataset(CONFIG['data_dir'], img_size=CONFIG['img_size'])
print(f'\n✅ Loaded {len(X)} images with shape {X.shape[1:]}')

# Class distribution analysis
class_counts = Counter(y)
print('\n📊 Class Distribution:')
for cls_idx, count in sorted(class_counts.items()):
    cls_name = CONFIG['class_display'][cls_idx]
    pct = count / len(y) * 100
    bar = '█' * int(pct // 2)
    print(f'  {cls_name:12s}: {bar} {count:4d} ({pct:.1f}%)')

# Stratified splits
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=CONFIG['test_ratio'], random_state=SEED, stratify=y
)
val_size = CONFIG['val_ratio'] / (1 - CONFIG['test_ratio'])
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size, random_state=SEED, stratify=y_temp
)

print(f'\n📊 Split Sizes:')
print(f'  Train: {len(X_train):,} ({100*len(X_train)/len(X):.1f}%)')
print(f'  Val:   {len(X_val):,} ({100*len(X_val)/len(X):.1f}%)')
print(f'  Test:  {len(X_test):,} ({100*len(X_test)/len(X):.1f}%)')

# Compute class weights for imbalanced training
class_weights_arr = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = dict(enumerate(class_weights_arr))
print(f'\n⚖️  Class Weights (for imbalanced training):')
for idx, w in class_weights.items():
    print(f'  {CONFIG["class_display"][idx]:12s}: {w:.3f}')

In [ ]:
# ── Visualize Sample Images ────────────────────────────────────────────────────
fig, axes = plt.subplots(4, 6, figsize=(20, 14))
fig.patch.set_facecolor('#0a0a0f')
fig.suptitle('🧠 Brain MRI Sample Gallery — All 4 Classes', 
             color='white', fontsize=16, fontweight='bold', y=1.01)

for row, cls_idx in enumerate(range(4)):
    cls_samples = X_train[y_train == cls_idx]
    
    # Class label on left
    axes[row, 0].text(0.5, 0.5, CONFIG['class_display'][cls_idx],
                      transform=axes[row, 0].transAxes,
                      ha='center', va='center', fontsize=13, fontweight='bold',
                      color=CONFIG['class_colors'][cls_idx],
                      rotation=0)
    axes[row, 0].axis('off')
    
    for col in range(1, 6):
        if col - 1 < len(cls_samples):
            img = cls_samples[col - 1]
            axes[row, col].imshow(img, cmap='gray' if img.mean() < 0.5 else None)
            axes[row, col].axis('off')
            
            # Colored border for class
            for spine in axes[row, col].spines.values():
                spine.set_edgecolor(CONFIG['class_colors'][cls_idx])
                spine.set_linewidth(2)
                spine.set_visible(True)

plt.tight_layout()
plt.savefig('sample_gallery.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
plt.show()

## 🏗️ Section 6: Model Architectures

### Architecture Overview

| Model | Parameters | ImageNet Top-1 | Depth | Notes |
|-------|-----------|----------------|-------|-------|
| Baseline CNN | ~500K | N/A (scratch) | 8 conv layers | Good baseline reference |
| ResNet50 | 25.6M | 74.9% | 50 layers | Skip connections, solid performer |
| EfficientNetB0 | 5.3M | 77.1% | Variable | Best accuracy/param ratio |
| MobileNetV2 | 3.4M | 71.8% | 53 layers | Fastest inference, mobile-ready |

### Transfer Learning Strategy
**Phase 1 — Feature Extraction** (frozen backbone):
- Freeze all pretrained layers
- Train only the classification head
- Higher learning rate (1e-4)
- Fast convergence, ~10-15 epochs

**Phase 2 — Fine-tuning** (unfrozen top layers):
- Unfreeze last N layers of backbone
- Very low learning rate (1e-5) to avoid forgetting
- Careful early stopping to prevent overfitting

In [ ]:
def build_baseline_cnn(input_shape: tuple, num_classes: int, dropout_rate: float = 0.5) -> Model:
    """
    Baseline CNN built from scratch.
    
    Architecture: 4 Conv blocks + Global Average Pooling + Dense classifier
    Each conv block: Conv2D → BatchNorm → ReLU → MaxPool → Dropout
    
    BatchNormalization:
    - Normalizes layer inputs during training
    - Reduces internal covariate shift
    - Acts as regularizer (reduces need for Dropout in some cases)
    - Allows higher learning rates
    
    Global Average Pooling (GAP):
    - Replaces Flatten + Dense
    - Averages each feature map to single value
    - Dramatically reduces parameters and overfitting
    - Naturally aligns with Grad-CAM visualization
    """
    inputs = layers.Input(shape=input_shape)
    x = inputs
    
    # Block 1: Low-level features (edges, textures)
    x = layers.Conv2D(32, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(32, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 2: Mid-level features (shapes, gradients)
    x = layers.Conv2D(64, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # Block 3: High-level features (parts, semantic)
    x = layers.Conv2D(128, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.3)(x)
    
    # Block 4: Abstract features
    x = layers.Conv2D(256, (3, 3), padding='same', activation=None)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    
    # Global Average Pooling + Classification head
    x = layers.GlobalAveragePooling2D()(x)  # (batch, 256)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)     # Training-time dropout for regularization
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name='Baseline_CNN')
    return model


def build_transfer_model(backbone_name: str, input_shape: tuple, 
                         num_classes: int, dropout_rate: float = 0.5) -> tuple:
    """
    Build a transfer learning model with specified backbone.
    
    Architecture:
    Pretrained Backbone (ImageNet weights, frozen)
         │
    Global Average Pooling
         │
    Dense(512, ReLU) + BatchNorm + Dropout
         │
    Dense(256, ReLU) + BatchNorm + Dropout
         │
    Dense(num_classes, Softmax)
    
    Args:
        backbone_name: 'resnet50', 'efficientnetb0', 'mobilenetv2'
    Returns:
        (model, backbone): Model with trainable head only
    """
    # Load pretrained backbone
    backbone_map = {
        'resnet50': ResNet50,
        'efficientnetb0': EfficientNetB0,
        'mobilenetv2': MobileNetV2,
    }
    BackboneClass = backbone_map[backbone_name.lower()]
    
    # Include_top=False: removes ImageNet classification head
    # We replace it with our 4-class head
    backbone = BackboneClass(
        include_top=False,
        weights='imagenet',
        input_shape=input_shape
    )
    
    # Phase 1: Freeze backbone
    backbone.trainable = False
    
    # Build classification head
    inputs = layers.Input(shape=input_shape)
    x = backbone(inputs, training=False)  # training=False: BN uses stored statistics
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate * 0.8)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs, outputs, name=f'TransferLearning_{backbone_name}')
    return model, backbone


# Build all models
print('🏗️ Building model architectures...')
input_shape = (CONFIG['img_size'], CONFIG['img_size'], CONFIG['channels'])
num_classes = len(CONFIG['classes'])

models_dict = {}
models_dict['Baseline CNN'] = (build_baseline_cnn(input_shape, num_classes), None)

for backbone in ['ResNet50', 'EfficientNetB0', 'MobileNetV2']:
    model, bb = build_transfer_model(backbone, input_shape, num_classes)
    models_dict[backbone] = (model, bb)

# Print model summaries
for name, (model, _) in models_dict.items():
    total_params = model.count_params()
    trainable = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    print(f'  {name:20s}: {total_params/1e6:.2f}M params | {trainable/1e6:.2f}M trainable')

## 🏋️ Section 7: Training with Callbacks & Learning Rate Scheduling

### Training Strategy

**Callbacks used:**
- **EarlyStopping**: Stops training when validation accuracy stops improving (patience=10). Prevents overfitting and saves training time.
- **ModelCheckpoint**: Saves the best model (by val_accuracy) to disk. Ensures we keep the best generalization, not the final epoch.
- **ReduceLROnPlateau**: Reduces learning rate by 50% when plateau detected. Allows fine-grained optimization near convergence.
- **TensorBoard**: Logs metrics for interactive visualization.

**Loss Function: Categorical Cross-Entropy**
```
L = -Σ y_true × log(y_pred)
```
With class weights, it becomes:
```
L = -Σ w_class × y_true × log(y_pred)
```

In [ ]:
def get_callbacks(model_name: str, monitor: str = 'val_accuracy') -> list:
    """Define training callbacks for a model."""
    return [
        callbacks.EarlyStopping(
            monitor=monitor,
            patience=10,
            restore_best_weights=True,  # Restores weights from best epoch
            verbose=1,
            min_delta=0.001
        ),
        callbacks.ModelCheckpoint(
            filepath=f'checkpoints/best_{model_name.replace(" ", "_")}.h5',
            monitor=monitor,
            save_best_only=True,
            verbose=0
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,           # Multiply LR by 0.5
            patience=5,           # Wait 5 epochs
            min_lr=1e-7,          # Floor
            verbose=1
        ),
        callbacks.CSVLogger(f'logs/{model_name.replace(" ", "_")}_history.csv')
    ]


def train_model(model_name: str, model: Model, backbone: Model,
                X_train, y_train, X_val, y_val,
                phase1_epochs: int = 15, phase2_epochs: int = 10) -> dict:
    """
    Two-phase transfer learning training.
    
    Phase 1 - Head only (fast convergence):
      Backbone frozen → only classification head trains
      LR = 1e-4, epochs = phase1_epochs
    
    Phase 2 - Fine-tuning (optional, for transfer models):
      Unfreeze top 30 layers of backbone
      LR = 1e-5 (10× smaller to preserve ImageNet features)
      epochs = phase2_epochs
    """
    os.makedirs('checkpoints', exist_ok=True)
    os.makedirs('logs', exist_ok=True)
    
    y_train_cat = to_categorical(y_train, num_classes)
    y_val_cat = to_categorical(y_val, num_classes)
    
    # ── Phase 1: Feature extraction ────────────────────────────────────────────
    print(f'\n🏋️ Phase 1: Training {model_name} (head only)...')
    model.compile(
        optimizer=optimizers.Adam(learning_rate=CONFIG['learning_rate']),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc', multi_label=False),
                 tf.keras.metrics.Precision(name='precision'),
                 tf.keras.metrics.Recall(name='recall')]
    )
    
    # Data augmentation generator
    train_datagen = ImageDataGenerator(
        rotation_range=20, width_shift_range=0.1,
        height_shift_range=0.1, zoom_range=0.1,
        horizontal_flip=True, brightness_range=[0.85, 1.15]
    )
    
    history1 = model.fit(
        train_datagen.flow(X_train, y_train_cat, batch_size=CONFIG['batch_size']),
        validation_data=(X_val, y_val_cat),
        epochs=phase1_epochs,
        class_weight=class_weights,
        callbacks=get_callbacks(f'{model_name}_phase1'),
        verbose=1
    )
    
    # ── Phase 2: Fine-tuning (skip for baseline CNN) ───────────────────────────
    history2 = None
    if backbone is not None and phase2_epochs > 0:
        print(f'\n🔬 Phase 2: Fine-tuning {model_name} (unfreeze top layers)...')
        
        # Unfreeze last 30 layers of backbone
        backbone.trainable = True
        for layer in backbone.layers[:-30]:
            layer.trainable = False
        
        # Much lower LR to avoid catastrophic forgetting
        model.compile(
            optimizer=optimizers.Adam(learning_rate=CONFIG['finetune_lr']),
            loss='categorical_crossentropy',
            metrics=['accuracy',
                     tf.keras.metrics.AUC(name='auc', multi_label=False)]
        )
        
        history2 = model.fit(
            train_datagen.flow(X_train, y_train_cat, batch_size=CONFIG['batch_size'] // 2),
            validation_data=(X_val, y_val_cat),
            epochs=phase2_epochs,
            class_weight=class_weights,
            callbacks=get_callbacks(f'{model_name}_phase2'),
            verbose=1
        )
    
    # Combine histories
    combined_history = {}
    for key in history1.history:
        vals = history1.history[key]
        if history2 and key in history2.history:
            vals = vals + history2.history[key]
        combined_history[key] = vals
    
    return combined_history


# Train all models
all_histories = {}

# For demo, use reduced epochs (increase for real training)
DEMO_MODE = True  # Set False for full training
epochs_p1 = 3 if DEMO_MODE else CONFIG['epochs_transfer']
epochs_p2 = 2 if DEMO_MODE else CONFIG['epochs_finetune']

for name, (model, backbone) in models_dict.items():
    print(f'\n{"="*60}')
    print(f'Training: {name}')
    print('='*60)
    p1 = epochs_p1 if name != 'Baseline CNN' else epochs_p1 * 2
    p2 = epochs_p2 if name != 'Baseline CNN' else 0
    
    history = train_model(name, model, backbone, X_train, y_train, X_val, y_val,
                          phase1_epochs=p1, phase2_epochs=p2)
    all_histories[name] = history
    
    final_val_acc = max(history.get('val_accuracy', [0]))
    print(f'  ✅ Best Val Accuracy: {final_val_acc:.4f}')

print('\n🎉 All models trained!')

## 📊 Section 8: Training Diagnostics & Learning Curves

Learning curves reveal critical information about model training:
- **Overfitting**: Train accuracy >> Val accuracy → add regularization
- **Underfitting**: Both accuracies low → increase model capacity
- **Convergence**: Both curves plateau → training complete
- **Learning rate issues**: Oscillating loss → LR too high
- **Phase 2 effect**: Jump in performance after fine-tuning unfreezes

In [ ]:
def plot_training_curves(histories: dict):
    """Comprehensive training diagnostics dashboard."""
    n_models = len(histories)
    fig, axes = plt.subplots(n_models, 3, figsize=(20, 5 * n_models))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle('📈 Training Diagnostics Dashboard', 
                 color='white', fontsize=16, fontweight='bold', y=1.01)
    
    if n_models == 1:
        axes = [axes]
    
    for row, (name, hist) in enumerate(histories.items()):
        color = CONFIG['class_colors'][row % 4]
        epochs = range(1, len(hist['accuracy']) + 1)
        
        # Plot 1: Accuracy
        ax = axes[row][0]
        ax.plot(epochs, hist['accuracy'], color=color, linewidth=2, label='Train')
        ax.plot(epochs, hist['val_accuracy'], color=color, linewidth=2, 
                linestyle='--', label='Validation', alpha=0.8)
        ax.fill_between(epochs, hist['accuracy'], hist['val_accuracy'],
                        alpha=0.1, color=color, label='Overfitting gap')
        ax.set_title(f'{name}\nAccuracy', color='white', fontweight='bold')
        ax.set_ylabel('Accuracy', color='white')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Plot 2: Loss
        ax = axes[row][1]
        ax.plot(epochs, hist['loss'], color='#e74c3c', linewidth=2, label='Train')
        ax.plot(epochs, hist['val_loss'], color='#e74c3c', linewidth=2,
                linestyle='--', label='Validation', alpha=0.8)
        ax.set_title(f'{name}\nLoss (Cross-Entropy)', color='white', fontweight='bold')
        ax.set_ylabel('Loss', color='white')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        # Plot 3: AUC (if available)
        ax = axes[row][2]
        if 'auc' in hist:
            ax.plot(epochs, hist['auc'], color='#2ecc71', linewidth=2, label='Train AUC')
            ax.plot(epochs, hist['val_auc'], color='#2ecc71', linewidth=2,
                    linestyle='--', label='Val AUC', alpha=0.8)
            ax.set_ylim(0, 1)
        ax.set_title(f'{name}\nAUC Score', color='white', fontweight='bold')
        ax.set_ylabel('AUC', color='white')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        for ax in axes[row]:
            ax.set_xlabel('Epoch', color='white')
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()

plot_training_curves(all_histories)

## 🎯 Section 9: Comprehensive Model Evaluation

### Evaluation Metrics for Medical AI

In medical imaging, accuracy alone is insufficient. We measure:

| Metric | Formula | Medical Relevance |
|--------|---------|-------------------|
| **Sensitivity/Recall** | TP/(TP+FN) | Ability to detect true positives (not miss tumors!) |
| **Specificity** | TN/(TN+FP) | Avoid false alarms (unnecessary biopsies) |
| **AUC-ROC** | Area under ROC curve | Overall discrimination ability |
| **F1-Score** | 2×P×R/(P+R) | Balance of precision & recall |

**For tumor detection, sensitivity is prioritized over specificity**: Missing a tumor (FN) is far more dangerous than a false alarm (FP).

In [ ]:
def evaluate_model_comprehensive(model: Model, X_test: np.ndarray, 
                                  y_test: np.ndarray, model_name: str) -> dict:
    """Full evaluation suite: accuracy, AUC, F1, confusion matrix, ROC curves."""
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_test_bin = label_binarize(y_test, classes=range(num_classes))
    
    # Metrics
    acc = np.mean(y_pred == y_test)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    try:
        auc_macro = roc_auc_score(y_test_bin, y_pred_prob, multi_class='ovr', average='macro')
    except:
        auc_macro = 0.0
    
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, target_names=CONFIG['class_display'], output_dict=True)
    
    # Per-class ROC curves
    fpr_dict, tpr_dict, auc_dict = {}, {}, {}
    for i, cls in enumerate(CONFIG['class_display']):
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_pred_prob[:, i])
        fpr_dict[cls] = fpr
        tpr_dict[cls] = tpr
        auc_dict[cls] = auc(fpr, tpr)
    
    result = {
        'accuracy': acc,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'auc_macro': auc_macro,
        'confusion_matrix': cm,
        'classification_report': report,
        'y_pred': y_pred,
        'y_pred_prob': y_pred_prob,
        'fpr': fpr_dict,
        'tpr': tpr_dict,
        'auc_per_class': auc_dict,
    }
    
    print(f'\n📊 {model_name} Results:')
    print(f'  Accuracy:    {acc:.4f}')
    print(f'  AUC (macro): {auc_macro:.4f}')
    print(f'  F1 (macro):  {f1_macro:.4f}')
    print(f'  F1 (weighted): {f1_weighted:.4f}')
    
    return result


# Evaluate all models
print('🔬 Evaluating all models on test set...')
eval_results = {}
for name, (model, _) in models_dict.items():
    eval_results[name] = evaluate_model_comprehensive(model, X_test, y_test, name)

# Best model identification
best_model_name = max(eval_results, key=lambda k: eval_results[k]['auc_macro'])
best_model = models_dict[best_model_name][0]
print(f'\n🏆 Best Model: {best_model_name} (AUC: {eval_results[best_model_name]["auc_macro"]:.4f})')

In [ ]:
# ── Visualization: Confusion Matrix & ROC Curves ──────────────────────────────
def plot_evaluation_dashboard(eval_results: dict):
    n = len(eval_results)
    fig = plt.figure(figsize=(22, 10 * ((n + 1) // 2)))
    fig.patch.set_facecolor('#0a0a0f')
    
    # Model comparison bar chart
    ax_compare = fig.add_subplot(2, n + 1, 1)
    names = list(eval_results.keys())
    accs = [eval_results[n]['accuracy'] for n in names]
    aucs = [eval_results[n]['auc_macro'] for n in names]
    f1s = [eval_results[n]['f1_macro'] for n in names]
    
    x = np.arange(len(names))
    w = 0.25
    ax_compare.bar(x - w, accs, w, label='Accuracy', color='#3498db', alpha=0.85)
    ax_compare.bar(x, aucs, w, label='AUC', color='#2ecc71', alpha=0.85)
    ax_compare.bar(x + w, f1s, w, label='F1', color='#e74c3c', alpha=0.85)
    ax_compare.set_xticks(x)
    ax_compare.set_xticklabels([n.replace(' ', '\n') for n in names], fontsize=9)
    ax_compare.set_ylim(0, 1.05)
    ax_compare.set_title('Model Comparison', color='white', fontweight='bold')
    ax_compare.legend(fontsize=9)
    ax_compare.grid(True, alpha=0.3, axis='y')
    
    # Confusion matrices
    for i, (name, result) in enumerate(eval_results.items()):
        ax = fig.add_subplot(2, n + 1, i + 2)
        cm = result['confusion_matrix']
        cm_pct = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
        im = ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=1)
        ax.set_xticks(range(num_classes))
        ax.set_xticklabels(CONFIG['class_display'], rotation=45, ha='right', fontsize=8, color='white')
        ax.set_yticks(range(num_classes))
        ax.set_yticklabels(CONFIG['class_display'], fontsize=8, color='white')
        
        for ii in range(cm.shape[0]):
            for jj in range(cm.shape[1]):
                ax.text(jj, ii, f'{cm[ii,jj]}\n{cm_pct[ii,jj]:.0%}',
                        ha='center', va='center', fontsize=8, fontweight='bold',
                        color='white' if cm_pct[ii,jj] < 0.6 else '#1a1a2e')
        
        star = ' ⭐' if name == best_model_name else ''
        ax.set_title(f'{name}{star}\nAcc: {result["accuracy"]:.3f}',
                     color='white', fontweight='bold', fontsize=10)
    
    # ROC Curves (best model)
    ax_roc = fig.add_subplot(2, n + 1, n + 2)
    best_result = eval_results[best_model_name]
    for i, cls in enumerate(CONFIG['class_display']):
        ax_roc.plot(best_result['fpr'][cls], best_result['tpr'][cls],
                    color=CONFIG['class_colors'][i], linewidth=2,
                    label=f'{cls} (AUC={best_result["auc_per_class"][cls]:.3f})')
    ax_roc.plot([0, 1], [0, 1], 'k--', alpha=0.5)
    ax_roc.set_title(f'ROC Curves — {best_model_name}', color='white', fontweight='bold')
    ax_roc.set_xlabel('False Positive Rate', color='white')
    ax_roc.set_ylabel('True Positive Rate', color='white')
    ax_roc.legend(fontsize=9)
    ax_roc.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('evaluation_dashboard.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()

plot_evaluation_dashboard(eval_results)

## 🔥 Section 10: Grad-CAM — Making the Model Explainable

### What is Grad-CAM?

**Gradient-weighted Class Activation Mapping** (Selvaraju et al., 2017) produces visual explanations for CNN decisions without architectural changes.

### How It Works (Step by Step)

1. **Select target layer**: Choose the last convolutional layer (richest spatial information)
2. **Forward pass**: Run image through model, get class score
3. **Backward pass**: Compute gradient of class score w.r.t. feature map activations
4. **Global Average Pooling**: Average gradients spatially → importance weight per channel
5. **Weighted sum**: Weight feature maps by channel importance
6. **ReLU**: Keep only positive contributions (what activates the class)
7. **Upsample**: Resize heatmap to original image size

**Formula:**
```
α_k^c = (1/Z) Σ_i Σ_j (∂y^c / ∂A_ij^k)   [importance of channel k for class c]

L_Grad-CAM^c = ReLU(Σ_k α_k^c × A^k)      [weighted combination of feature maps]
```

### Clinical Significance
Grad-CAM allows radiologists to:
- Verify the model is looking at the tumor, not artifacts
- Build trust through transparency
- Identify failure modes (model focusing on non-relevant regions)

In [ ]:
class GradCAM:
    """
    Grad-CAM implementation for TensorFlow/Keras models.
    
    Supports both:
    - Standard Grad-CAM: gradient-weighted feature activation map
    - Guided Grad-CAM: Grad-CAM × guided backpropagation (sharper)
    """
    
    def __init__(self, model: Model, layer_name: str = None):
        """
        Args:
            model: Trained Keras model
            layer_name: Name of target conv layer. If None, finds last conv layer.
        """
        self.model = model
        self.layer_name = layer_name or self._find_last_conv_layer()
        print(f'  🎯 Grad-CAM target layer: {self.layer_name}')
    
    def _find_last_conv_layer(self) -> str:
        """Automatically find the last convolutional layer."""
        for layer in reversed(self.model.layers):
            if isinstance(layer, (layers.Conv2D,)) or \
               hasattr(layer, 'output') and len(getattr(layer, 'output_shape', (None,))) == 4:
                return layer.name
        raise ValueError('No convolutional layer found in model!')
    
    def compute_heatmap(self, image: np.ndarray, class_idx: int = None,
                         eps: float = 1e-8) -> tuple:
        """
        Compute Grad-CAM heatmap for an input image.
        
        Args:
            image: Preprocessed image (H, W, 3) or (1, H, W, 3)
            class_idx: Target class. If None, uses predicted class.
            eps: Small value to prevent division by zero
            
        Returns:
            heatmap: Raw Grad-CAM heatmap (14×14 for most models)
            upsampled: Heatmap resized to input image dimensions
            class_idx: The class used for Grad-CAM
            pred_probs: Model's probability output
        """
        if image.ndim == 3:
            image = np.expand_dims(image, 0)  # Add batch dimension
        
        # Build gradient model: outputs both conv activations and predictions
        try:
            target_layer = self.model.get_layer(self.layer_name)
        except ValueError:
            # Try to find through sub-models
            for layer in self.model.layers:
                if hasattr(layer, 'layers'):
                    try:
                        target_layer = layer.get_layer(self.layer_name)
                        break
                    except ValueError:
                        continue
        
        grad_model = Model(
            inputs=self.model.inputs,
            outputs=[target_layer.output, self.model.output]
        )
        
        # Compute gradients with GradientTape
        with tf.GradientTape() as tape:
            img_tensor = tf.cast(image, tf.float32)
            tape.watch(img_tensor)
            conv_outputs, predictions = grad_model(img_tensor)
            
            if class_idx is None:
                class_idx = np.argmax(predictions[0])
            
            # Score for target class
            loss = predictions[:, class_idx]
        
        # Gradient of class score w.r.t. feature map
        grads = tape.gradient(loss, conv_outputs)
        
        # Global Average Pooling of gradients → importance weights
        pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
        
        # Weight feature maps by importance
        conv_outputs = conv_outputs[0]  # Remove batch dim
        heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
        heatmap = tf.squeeze(heatmap)   # (H, W)
        
        # ReLU: keep only positive influences
        heatmap = tf.maximum(heatmap, 0)
        
        # Normalize to [0, 1]
        heatmap = (heatmap - tf.reduce_min(heatmap)) / \
                  (tf.reduce_max(heatmap) - tf.reduce_min(heatmap) + eps)
        heatmap = heatmap.numpy()
        
        # Upsample to input size
        input_h, input_w = image.shape[1], image.shape[2]
        upsampled = cv2.resize(heatmap, (input_w, input_h))
        upsampled = np.clip(upsampled, 0, 1)
        
        return heatmap, upsampled, class_idx, predictions[0].numpy()
    
    def overlay_heatmap(self, image: np.ndarray, heatmap: np.ndarray,
                         alpha: float = 0.5, colormap: int = cv2.COLORMAP_JET) -> np.ndarray:
        """
        Overlay Grad-CAM heatmap on original image.
        
        Args:
            alpha: Blending factor (0=original only, 1=heatmap only)
        """
        # Convert heatmap to colormap
        heatmap_uint8 = (heatmap * 255).astype(np.uint8)
        heatmap_color = cv2.applyColorMap(heatmap_uint8, colormap)
        
        # Convert image to uint8
        if image.dtype == np.float32 or image.dtype == np.float64:
            image_uint8 = (image * 255).astype(np.uint8)
        else:
            image_uint8 = image.copy()
        
        # Ensure 3 channels
        if image_uint8.ndim == 2:
            image_uint8 = cv2.cvtColor(image_uint8, cv2.COLOR_GRAY2BGR)
        
        # Blend
        overlay = cv2.addWeighted(image_uint8, 1 - alpha, heatmap_color, alpha, 0)
        return cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)


print('✅ GradCAM class defined!')

In [ ]:
def visualize_gradcam_comprehensive(model: Model, X_samples: np.ndarray, 
                                     y_samples: np.ndarray, 
                                     layer_name: str = None,
                                     n_per_class: int = 2):
    """
    Create comprehensive Grad-CAM visualization grid showing:
    - Original MRI scan
    - Grad-CAM heatmap only
    - Overlay (MRI + Grad-CAM)
    - Probability bar chart
    
    For each class, shows examples for both correct and incorrect predictions.
    """
    gradcam = GradCAM(model, layer_name)
    
    n_classes = len(CONFIG['classes'])
    fig, axes = plt.subplots(n_classes * n_per_class, 4,
                              figsize=(18, 5 * n_classes * n_per_class))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle('🔥 Grad-CAM Explainability — What the Model Sees',
                 color='white', fontsize=16, fontweight='bold', y=1.01)
    
    col_titles = ['Original MRI', 'Grad-CAM Heatmap', 'Overlay', 'Class Probabilities']
    for col, title in enumerate(col_titles):
        axes[0, col].set_title(title, color='white', fontsize=12, fontweight='bold', pad=15)
    
    row = 0
    for cls_idx in range(n_classes):
        cls_mask = y_samples == cls_idx
        cls_imgs = X_samples[cls_mask][:n_per_class]
        cls_labels = y_samples[cls_mask][:n_per_class]
        
        if len(cls_imgs) == 0:
            row += n_per_class
            continue
        
        for sample_idx, (img, true_label) in enumerate(zip(cls_imgs, cls_labels)):
            # Compute Grad-CAM
            _, upsampled_heatmap, pred_class, pred_probs = gradcam.compute_heatmap(img)
            correct = pred_class == true_label
            
            # Column 1: Original
            axes[row, 0].imshow(img, cmap='gray')
            status = '✓' if correct else '✗'
            axes[row, 0].set_ylabel(
                f'{CONFIG["class_display"][cls_idx]}\n{status} Pred: {CONFIG["class_display"][pred_class]}',
                color=CONFIG['class_colors'][cls_idx], fontsize=9, fontweight='bold'
            )
            axes[row, 0].axis('off')
            
            # Column 2: Heatmap only
            axes[row, 1].imshow(upsampled_heatmap, cmap='jet', vmin=0, vmax=1)
            axes[row, 1].axis('off')
            
            # Column 3: Overlay
            overlay = gradcam.overlay_heatmap(img, upsampled_heatmap, alpha=0.45)
            axes[row, 2].imshow(overlay)
            axes[row, 2].axis('off')
            
            # Column 4: Probability bar chart
            bars = axes[row, 3].barh(
                CONFIG['class_display'], pred_probs,
                color=[CONFIG['class_colors'][i] if i == pred_class else '#3a3a5c'
                       for i in range(n_classes)],
                edgecolor='#555', linewidth=0.5
            )
            axes[row, 3].set_xlim(0, 1)
            axes[row, 3].axvline(x=0.5, color='white', linestyle='--', alpha=0.4)
            for bar, prob in zip(bars, pred_probs):
                axes[row, 3].text(min(prob + 0.02, 0.95), bar.get_y() + bar.get_height() / 2,
                                   f'{prob:.1%}', va='center', ha='left',
                                   color='white', fontsize=8, fontweight='bold')
            axes[row, 3].set_xlabel('Probability', color='white', fontsize=9)
            axes[row, 3].grid(True, axis='x', alpha=0.3)
            
            # Border color indicates correct/incorrect
            border_color = '#2ecc71' if correct else '#e74c3c'
            for ax in axes[row]:
                for spine in ax.spines.values():
                    spine.set_edgecolor(border_color)
                    spine.set_linewidth(2)
                    spine.set_visible(True)
            
            row += 1
    
    plt.tight_layout()
    plt.savefig('gradcam_visualization.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    print('💾 Saved: gradcam_visualization.png')


# Run Grad-CAM on best model
print(f'🔥 Generating Grad-CAM for {best_model_name}...')
# Select diverse samples from test set
n_samples = min(len(X_test), 50)
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)
X_samples = X_test[sample_indices]
y_samples = y_test[sample_indices]

visualize_gradcam_comprehensive(
    best_model, X_samples, y_samples,
    n_per_class=2
)

## 🎲 Section 11: Monte Carlo Dropout — Uncertainty Estimation

### Why Uncertainty Matters in Medical AI

A model that says "99% confident: Glioma" when it's wrong is dangerous. We want models that know what they don't know.

### Monte Carlo Dropout (Gal & Ghahramani, 2016)

**Key insight**: Dropout during inference approximates Bayesian uncertainty.

**Algorithm:**
1. Keep dropout layers **active during inference** (not standard practice)
2. Run N stochastic forward passes (different dropout masks each time)
3. Collect N probability distributions
4. **Mean**: Best prediction
5. **Std/Entropy**: Uncertainty estimate

**Interpretation:**
- Low uncertainty + correct → confident and right ✅
- High uncertainty → model is unsure, flag for expert review 🔍
- Low uncertainty + wrong → overconfident failure ⚠️ (most dangerous)

In [ ]:
def mc_dropout_predict(model: Model, image: np.ndarray,
                        n_samples: int = 50) -> dict:
    """
    Monte Carlo Dropout uncertainty estimation.
    
    Runs N stochastic forward passes with dropout ACTIVE during inference.
    This approximates a Bayesian Neural Network posterior.
    
    Args:
        image: Single preprocessed image (H, W, C) or (1, H, W, C)
        n_samples: Number of MC samples (more = better estimate, slower)
    
    Returns dict with:
        mean_probs: Mean of N predictions
        std_probs: Standard deviation (epistemic uncertainty)
        entropy: Information entropy of mean prediction
        all_preds: All N probability vectors
        predicted_class: Argmax of mean
        confidence: Max probability in mean prediction
        uncertainty: 1 - confidence (simple uncertainty)
    """
    if image.ndim == 3:
        image = np.expand_dims(image, 0)
    
    all_predictions = []
    
    for _ in range(n_samples):
        # training=True forces dropout to be ACTIVE during inference
        # This is the key difference from standard inference
        pred = model(image, training=True)  # Stochastic forward pass
        all_predictions.append(pred.numpy()[0])
    
    all_predictions = np.array(all_predictions)  # (n_samples, n_classes)
    
    mean_probs = all_predictions.mean(axis=0)
    std_probs = all_predictions.std(axis=0)
    
    # Predictive entropy: H = -Σ p(c) log p(c)
    entropy = -np.sum(mean_probs * np.log(mean_probs + 1e-8))
    max_entropy = np.log(len(mean_probs))  # Maximum possible entropy
    normalized_entropy = entropy / max_entropy  # Scale to [0, 1]
    
    predicted_class = np.argmax(mean_probs)
    confidence = mean_probs[predicted_class]
    
    return {
        'mean_probs': mean_probs,
        'std_probs': std_probs,
        'entropy': normalized_entropy,
        'all_preds': all_predictions,
        'predicted_class': predicted_class,
        'confidence': float(confidence),
        'uncertainty': float(1 - confidence),
        'is_certain': normalized_entropy < 0.3,  # Threshold for clinical use
    }


def visualize_uncertainty(model: Model, X_samples: np.ndarray, y_samples: np.ndarray):
    """Visualize MC Dropout uncertainty across samples."""
    print(f'🎲 Computing MC Dropout uncertainty ({CONFIG["mc_samples"]} samples each)...')
    
    results = []
    for img, true_label in tqdm(zip(X_samples[:20], y_samples[:20])):
        mc_result = mc_dropout_predict(model, img, n_samples=CONFIG['mc_samples'])
        mc_result['true_label'] = true_label
        mc_result['correct'] = mc_result['predicted_class'] == true_label
        results.append(mc_result)
    
    # Uncertainty distribution analysis
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle('🎲 Monte Carlo Dropout — Uncertainty Analysis', 
                 color='white', fontsize=14, fontweight='bold')
    
    # Entropy by correctness
    correct_entropy = [r['entropy'] for r in results if r['correct']]
    wrong_entropy = [r['entropy'] for r in results if not r['correct']]
    
    axes[0].hist(correct_entropy, bins=15, alpha=0.7, color='#2ecc71', label='Correct')
    axes[0].hist(wrong_entropy, bins=15, alpha=0.7, color='#e74c3c', label='Wrong')
    axes[0].set_title('Uncertainty Entropy\nby Prediction Correctness', color='white', fontweight='bold')
    axes[0].set_xlabel('Normalized Entropy', color='white')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Confidence distribution
    confidences = [r['confidence'] for r in results]
    colors = ['#2ecc71' if r['correct'] else '#e74c3c' for r in results]
    axes[1].bar(range(len(confidences)), confidences, color=colors, alpha=0.8)
    axes[1].axhline(y=0.5, color='white', linestyle='--', alpha=0.5)
    axes[1].set_title('Confidence per Sample', color='white', fontweight='bold')
    axes[1].set_ylabel('Confidence', color='white')
    axes[1].set_xlabel('Sample Index', color='white')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    # MC sample spread for one example
    sample_result = results[0]
    all_preds = sample_result['all_preds']  # (n_samples, n_classes)
    for i, cls in enumerate(CONFIG['class_display']):
        axes[2].hist(all_preds[:, i], bins=20, alpha=0.6,
                     color=CONFIG['class_colors'][i], label=cls)
    axes[2].set_title(f'MC Dropout Sample Distribution\nTrue: {CONFIG["class_display"][sample_result["true_label"]]}',
                      color='white', fontweight='bold')
    axes[2].set_xlabel('Predicted Probability', color='white')
    axes[2].set_ylabel('Frequency', color='white')
    axes[2].legend(fontsize=8)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('uncertainty_analysis.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    
    return results


uncertainty_results = visualize_uncertainty(best_model, X_test, y_test)

## 🗺️ Section 12: t-SNE Feature Space Visualization

**t-SNE (t-distributed Stochastic Neighbor Embedding)** is a dimensionality reduction technique that maps high-dimensional feature vectors to 2D while preserving local neighborhood structure.

### What We're Visualizing
We extract the **penultimate layer** (before classification head) features — these are the learned representations the model uses to distinguish tumor types. Good representations show:
- **Tight clusters**: Model learned class-discriminative features
- **Separated clusters**: Different classes are well-separated
- **No overlapping**: Model can distinguish between them

In [ ]:
def extract_features(model: Model, X: np.ndarray) -> np.ndarray:
    """Extract penultimate layer features for t-SNE visualization."""
    # Find the last dense layer before output
    feature_model = Model(
        inputs=model.input,
        outputs=model.layers[-2].output  # Second-to-last layer
    )
    return feature_model.predict(X, verbose=0, batch_size=32)


def visualize_tsne(model: Model, X: np.ndarray, y: np.ndarray, title: str = 'Feature Space'):
    """Create t-SNE visualization of learned feature space."""
    print('🗺️ Extracting features and computing t-SNE...')
    features = extract_features(model, X)
    
    # t-SNE: reduce to 2D
    # Perplexity: balance between local and global structure (typically 5-50)
    # n_iter: more iterations = better convergence
    tsne = TSNE(n_components=2, perplexity=30, n_iter=1000, 
                random_state=SEED, verbose=0)
    embedded = tsne.fit_transform(features)
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle(f'🗺️ t-SNE Feature Visualization — {title}',
                 color='white', fontsize=14, fontweight='bold')
    
    # Color by true class
    for cls_idx, (cls_name, color) in enumerate(zip(CONFIG['class_display'], CONFIG['class_colors'])):
        mask = y == cls_idx
        axes[0].scatter(embedded[mask, 0], embedded[mask, 1],
                        c=color, label=cls_name, s=25, alpha=0.7)
    axes[0].set_title('Colored by True Class', color='white', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.2)
    axes[0].set_xlabel('t-SNE Dim 1', color='white')
    axes[0].set_ylabel('t-SNE Dim 2', color='white')
    
    # Color by prediction confidence
    probs = model.predict(X, verbose=0, batch_size=32)
    confidence = probs.max(axis=1)
    sc = axes[1].scatter(embedded[:, 0], embedded[:, 1],
                          c=confidence, cmap='RdYlGn', s=25, alpha=0.7, vmin=0, vmax=1)
    plt.colorbar(sc, ax=axes[1], label='Prediction Confidence')
    axes[1].set_title('Colored by Model Confidence', color='white', fontweight='bold')
    axes[1].grid(True, alpha=0.2)
    axes[1].set_xlabel('t-SNE Dim 1', color='white')
    axes[1].set_ylabel('t-SNE Dim 2', color='white')
    
    plt.tight_layout()
    plt.savefig('tsne_visualization.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()


# Sample for t-SNE (all test samples)
visualize_tsne(best_model, X_test, y_test, best_model_name)

## 🤝 Section 13: Ensemble Model Fusion

**Ensemble methods** combine predictions from multiple models to improve robustness and accuracy. Three common strategies:

1. **Simple Average**: Average probabilities across all models
2. **Weighted Average**: Weight by validation accuracy (better models contribute more)
3. **Maximum Voting**: Final prediction = most common prediction

Ensembles work because different architectures make different errors — when they disagree, the majority is usually right.

In [ ]:
def ensemble_predict(models_dict: dict, eval_results: dict, 
                     X: np.ndarray, method: str = 'weighted') -> tuple:
    """
    Ensemble multiple models for improved predictions.
    
    Methods:
      'simple': Average of all model probabilities
      'weighted': Weight by validation accuracy
      'voting': Hard majority vote
    """
    all_probs = []
    weights = []
    
    for name, (model, _) in models_dict.items():
        probs = model.predict(X, verbose=0, batch_size=32)
        all_probs.append(probs)
        weights.append(eval_results[name]['accuracy'])
    
    all_probs = np.array(all_probs)  # (n_models, n_samples, n_classes)
    weights = np.array(weights)
    weights = weights / weights.sum()  # Normalize
    
    if method == 'simple':
        ensemble_probs = all_probs.mean(axis=0)
    elif method == 'weighted':
        ensemble_probs = np.average(all_probs, axis=0, weights=weights)
    elif method == 'voting':
        votes = all_probs.argmax(axis=2)  # (n_models, n_samples)
        ensemble_preds = []
        for i in range(votes.shape[1]):
            counts = np.bincount(votes[:, i], minlength=num_classes)
            ensemble_preds.append(counts.argmax())
        ensemble_preds = np.array(ensemble_preds)
        return ensemble_preds, None
    
    ensemble_preds = np.argmax(ensemble_probs, axis=1)
    return ensemble_preds, ensemble_probs


# Compare ensemble vs individual models
print('🤝 Computing ensemble predictions...')
ensemble_preds, ensemble_probs = ensemble_predict(models_dict, eval_results, X_test, method='weighted')
ensemble_acc = np.mean(ensemble_preds == y_test)

print('\n📊 Model Performance Summary:')
print(f'{"Model":<25} {"Accuracy":>10} {"AUC":>10} {"F1":>10}')
print('-' * 58)
for name, result in eval_results.items():
    print(f'{name:<25} {result["accuracy"]:>10.4f} {result["auc_macro"]:>10.4f} {result["f1_macro"]:>10.4f}')
print('-' * 58)
print(f'{"Ensemble (Weighted)":<25} {ensemble_acc:>10.4f} {"N/A":>10} {"N/A":>10}')

## 💾 Section 14: Model Export & Deployment Preparation

We export the best model in multiple formats for different deployment targets:
- **`.h5`**: Standard Keras format for Python deployment
- **`SavedModel`**: TensorFlow's native format for serving
- **`TFLite`**: Optimized for mobile/edge devices (2-5× smaller)

In [ ]:
import pickle

# Save best model
os.makedirs('models', exist_ok=True)

best_model.save(f'models/best_model_{best_model_name.replace(" ", "_")}.h5')
print(f'✅ Saved model: best_model_{best_model_name}.h5')

# Save training metadata
metadata = {
    'best_model': best_model_name,
    'classes': CONFIG['classes'],
    'class_display': CONFIG['class_display'],
    'img_size': CONFIG['img_size'],
    'results': {
        name: {
            'accuracy': float(r['accuracy']),
            'auc': float(r['auc_macro']),
            'f1_macro': float(r['f1_macro']),
        }
        for name, r in eval_results.items()
    },
    'ensemble_accuracy': float(ensemble_acc),
}
with open('models/metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Saved metadata: models/metadata.json')

# TFLite export for edge deployment
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Post-training quantization
tflite_model = converter.convert()
with open('models/best_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'✅ TFLite model: {len(tflite_model)/1e6:.1f}MB')

print('\n🎉 Training complete! Summary:')
print(f'  Best model: {best_model_name}')
print(f'  Test accuracy: {eval_results[best_model_name]["accuracy"]:.4f}')
print(f'  AUC macro: {eval_results[best_model_name]["auc_macro"]:.4f}')
print(f'  Ensemble accuracy: {ensemble_acc:.4f}')
print('\n→ Launch Streamlit app: streamlit run app.py')

## 🩻 Section 15: Per-Class Error Analysis & Misclassification Insights

Understanding *which* samples the model gets wrong is as important as overall accuracy.
We'll investigate:
- **Confusion patterns**: Which tumor pairs are most confused?
- **Confidence on errors**: Are wrong predictions overconfident?
- **Visual inspection**: Do misclassified images look ambiguous even to humans?

This analysis directly informs clinical deployment decisions — e.g., if glioma is consistently
confused with meningioma, a two-stage classifier or radiologist review gate may be warranted.

In [ ]:
def error_analysis(model, X_test, y_test, n_show=12):
    """
    Deep dive into model errors:
    1. Find all misclassified samples
    2. Rank by prediction confidence (most overconfident errors first)
    3. Show original image + predicted vs true label + confidence
    """
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    confidence = y_pred_prob.max(axis=1)

    # Find errors
    error_mask = (y_pred != y_test)
    error_indices = np.where(error_mask)[0]

    if len(error_indices) == 0:
        print('🎉 No errors found on test set!')
        return

    # Sort by confidence (most overconfident first — most dangerous)
    sorted_idx = error_indices[np.argsort(confidence[error_indices])[::-1]]

    print(f'❌ Total errors: {len(error_indices)}/{len(y_test)} ({100*len(error_indices)/len(y_test):.1f}%)')
    print(f'\n🔍 Confusion Pattern:')
    confusion_pairs = {}
    for idx in error_indices:
        pair = (CONFIG['class_display'][y_test[idx]], CONFIG['class_display'][y_pred[idx]])
        confusion_pairs[pair] = confusion_pairs.get(pair, 0) + 1
    for (true_cls, pred_cls), count in sorted(confusion_pairs.items(), key=lambda x: -x[1]):
        print(f'  {true_cls:12s} → {pred_cls:12s}: {count:3d} times')

    # Plot worst errors
    n_show = min(n_show, len(sorted_idx))
    ncols = 4
    nrows = (n_show + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4.5 * nrows))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle('🔍 Most Overconfident Errors (Sorted by Confidence)',
                 color='white', fontsize=14, fontweight='bold')
    axes = axes.flatten() if nrows > 1 else [axes] if ncols == 1 else axes.flatten()

    for plot_i, idx in enumerate(sorted_idx[:n_show]):
        ax = axes[plot_i]
        ax.imshow(X_test[idx], cmap='gray')

        true_cls = CONFIG['class_display'][y_test[idx]]
        pred_cls = CONFIG['class_display'][y_pred[idx]]
        conf = confidence[idx]

        ax.set_title(f'True: {true_cls}\nPred: {pred_cls} ({conf:.1%})',
                     color='#ff6b6b', fontsize=8.5, fontweight='bold', pad=6)
        ax.axis('off')

        for spine in ax.spines.values():
            spine.set_edgecolor('#e74c3c')
            spine.set_linewidth(2)
            spine.set_visible(True)

    for ax in axes[n_show:]:
        ax.axis('off')

    plt.tight_layout()
    plt.savefig('error_analysis.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    print('💾 Saved: error_analysis.png')

error_analysis(best_model, X_test, y_test)

## 📐 Section 16: Clinical Threshold Calibration

### The Problem with Default Thresholds

Standard models predict by taking `argmax(probabilities)`, implicitly using a 50% threshold.
In medical imaging this is suboptimal because:
- **Asymmetric costs**: Missing a tumor (FN) is far more dangerous than a false alarm (FP)
- **Class imbalance**: Rare classes need lower thresholds to be detected

### Threshold Optimization

For each class we find the threshold that maximizes the **F2 score** (weights recall 2× more than precision):
```
F2 = 5 × Precision × Recall / (4 × Precision + Recall)
```

### Probability Calibration

Raw neural network probabilities are often overconfident (outputs near 0 or 1 even when uncertain).
We use **Temperature Scaling** — divides logits by temperature T before softmax:
- T > 1 → softer, more calibrated probabilities
- T < 1 → sharper, more confident probabilities
- T = 1 → no change (default)

In [ ]:
def calibrate_model_temperature(model, X_val, y_val, T_range=np.linspace(0.5, 3.0, 50)):
    """
    Temperature scaling calibration.
    Finds temperature T that minimizes Expected Calibration Error (ECE)
    on the validation set.

    ECE = Σ |acc(Bm) - conf(Bm)| × |Bm| / n
    where Bm are confidence bins.
    """
    probs_val = model.predict(X_val, verbose=0)

    def compute_ece(probs, y_true, n_bins=10):
        """Expected Calibration Error — measures probability reliability."""
        bins = np.linspace(0, 1, n_bins + 1)
        ece = 0.0
        for b in range(n_bins):
            mask = (probs.max(axis=1) >= bins[b]) & (probs.max(axis=1) < bins[b+1])
            if mask.sum() == 0:
                continue
            acc = (probs[mask].argmax(axis=1) == y_true[mask]).mean()
            conf = probs[mask].max(axis=1).mean()
            ece += abs(acc - conf) * mask.sum() / len(y_true)
        return ece

    best_T, best_ece = 1.0, float('inf')
    ece_scores = []

    for T in T_range:
        # Apply temperature scaling
        logits = np.log(probs_val + 1e-8)
        scaled = np.exp(logits / T)
        scaled_probs = scaled / scaled.sum(axis=1, keepdims=True)
        ece = compute_ece(scaled_probs, y_val)
        ece_scores.append(ece)
        if ece < best_ece:
            best_ece, best_T = ece, T

    # Uncalibrated ECE
    raw_ece = compute_ece(probs_val, y_val)

    # Plot temperature vs ECE
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0a0a0f')

    axes[0].plot(T_range, ece_scores, color='#3498db', linewidth=2)
    axes[0].axvline(x=best_T, color='#2ecc71', linestyle='--', linewidth=2,
                    label=f'Best T={best_T:.2f}')
    axes[0].axhline(y=raw_ece, color='#e74c3c', linestyle=':', linewidth=1.5,
                    label=f'Uncalibrated ECE={raw_ece:.4f}')
    axes[0].set_xlabel('Temperature T', color='white')
    axes[0].set_ylabel('Expected Calibration Error (ECE)', color='white')
    axes[0].set_title('Temperature Scaling Calibration', color='white', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Reliability diagram (calibration curve)
    logits = np.log(probs_val + 1e-8)
    scaled = np.exp(logits / best_T)
    cal_probs = scaled / scaled.sum(axis=1, keepdims=True)

    n_bins = 10
    bins = np.linspace(0, 1, n_bins + 1)
    bin_accs, bin_confs, bin_counts = [], [], []

    for b in range(n_bins):
        mask = (cal_probs.max(axis=1) >= bins[b]) & (cal_probs.max(axis=1) < bins[b+1])
        if mask.sum() > 0:
            bin_accs.append((cal_probs[mask].argmax(axis=1) == y_val[mask]).mean())
            bin_confs.append(cal_probs[mask].max(axis=1).mean())
            bin_counts.append(mask.sum())

    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect calibration')
    axes[1].scatter(bin_confs, bin_accs, s=np.array(bin_counts)*3, 
                    color='#3498db', alpha=0.8, zorder=5, label='Calibrated')
    axes[1].plot(bin_confs, bin_accs, color='#3498db', linewidth=2)
    axes[1].fill_between(bin_confs, bin_accs, bin_confs, alpha=0.2, color='#e74c3c', label='Gap')
    axes[1].set_xlabel('Confidence', color='white')
    axes[1].set_ylabel('Accuracy', color='white')
    axes[1].set_title(f'Reliability Diagram\nBest T={best_T:.2f}, ECE={best_ece:.4f}',
                      color='white', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('calibration.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()

    print(f'✅ Calibration results:')
    print(f'  Best Temperature:    T = {best_T:.3f}')
    print(f'  Uncalibrated ECE:    {raw_ece:.4f}')
    print(f'  Calibrated ECE:      {best_ece:.4f}')
    print(f'  Improvement:         {100*(raw_ece-best_ece)/raw_ece:.1f}%')
    return best_T

best_temperature = calibrate_model_temperature(best_model, X_val, y_val)

## 🔬 Section 17: Guided Backpropagation + Grad-CAM Fusion

### Guided Backpropagation
Standard backpropagation propagates both positive and negative gradients.
**Guided Backpropagation** (Springenberg et al., 2014) only propagates gradients where:
1. The gradient is positive (standard ReLU)
2. **AND** the activation was positive (additional guidance gate)

This produces much sharper, more neuron-specific saliency maps.

### Grad-CAM++ (Upgraded)
**Grad-CAM++** (Chattopadhyay et al., 2018) improves upon standard Grad-CAM:
- Better localization when multiple instances of same class appear
- Pixel-wise weighting (not just channel-wise)
- More faithful to model's actual attention

Combined: `Guided Grad-CAM = Guided Backprop × Grad-CAM` gives both spatial precision AND semantic relevance.

In [ ]:
def compute_gradcam_plus_plus(model, image, class_idx=None, layer_name=None, eps=1e-8):
    """
    Grad-CAM++ implementation.
    Improvement over standard Grad-CAM: uses second-order gradients
    for more accurate pixel-wise importance weighting.

    Reference: Chattopadhyay et al. (2018) https://arxiv.org/abs/1710.11063
    """
    if image.ndim == 3:
        image = np.expand_dims(image, 0)

    # Find last conv layer if not specified
    if layer_name is None:
        for layer in reversed(model.layers):
            if 'conv' in layer.name.lower():
                layer_name = layer.name
                break

    try:
        target_layer = model.get_layer(layer_name)
    except:
        for layer in model.layers:
            if hasattr(layer, 'layers'):
                try:
                    target_layer = layer.get_layer(layer_name)
                    break
                except: continue

    grad_model = Model(inputs=model.inputs,
                       outputs=[target_layer.output, model.output])

    with tf.GradientTape() as tape2:
        with tf.GradientTape() as tape1:
            with tf.GradientTape() as tape0:
                img_tensor = tf.cast(image, tf.float32)
                conv_output, preds = grad_model(img_tensor)
                if class_idx is None:
                    class_idx = np.argmax(preds[0])
                score = preds[:, class_idx]
            # First-order gradients
            grads1 = tape0.gradient(score, conv_output)
        # Second-order gradients
        grads2 = tape1.gradient(grads1, conv_output)
    # Third-order gradients
    grads3 = tape2.gradient(grads2, conv_output)

    # Grad-CAM++ weights
    sum_act = tf.reduce_sum(conv_output, axis=[1, 2])  # Global sum
    # Alpha: pixel-wise importance weights
    alpha_num = grads2
    alpha_denom = 2.0 * grads2 + sum_act[:, tf.newaxis, tf.newaxis, :] * grads3 + eps
    alpha = alpha_num / alpha_denom

    # Weight = ReLU(alpha) * ReLU(gradients)
    weights = tf.reduce_sum(tf.nn.relu(grads1) * alpha, axis=[1, 2])

    # Weighted feature map combination
    cam = tf.reduce_sum(conv_output[0] * weights[0], axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam.numpy()

    # Normalize and upsample
    cam = (cam - cam.min()) / (cam.max() - cam.min() + eps)
    cam = cv2.resize(cam, (image.shape[2], image.shape[1]))

    return cam, class_idx, preds[0].numpy()


def compare_gradcam_variants(model, X_test, y_test, n_samples=3):
    """
    Side-by-side comparison: Standard Grad-CAM vs Grad-CAM++
    Shows how Grad-CAM++ provides sharper, more precise localization.
    """
    gradcam_std = GradCAM(model)

    sample_indices = []
    for cls in range(len(CONFIG['classes'])):
        cls_idx = np.where(y_test == cls)[0]
        if len(cls_idx) > 0:
            sample_indices.append(cls_idx[0])

    fig, axes = plt.subplots(len(sample_indices), 5,
                              figsize=(22, 4.5 * len(sample_indices)))
    fig.patch.set_facecolor('#0a0a0f')
    fig.suptitle('🔬 Grad-CAM vs Grad-CAM++ Comparison',
                 color='white', fontsize=14, fontweight='bold')

    col_labels = ['Original', 'Grad-CAM', 'Grad-CAM Overlay', 'Grad-CAM++', 'Grad-CAM++ Overlay']
    for col, label in enumerate(col_labels):
        axes[0, col].set_title(label, color='white', fontsize=11, fontweight='bold')

    for row, idx in enumerate(sample_indices):
        img = X_test[idx]
        true_cls = CONFIG['class_display'][y_test[idx]]

        # Standard Grad-CAM
        _, cam_std, pred_cls, probs = gradcam_std.compute_heatmap(img)
        overlay_std = gradcam_std.overlay_heatmap(img, cam_std, alpha=0.5)

        # Grad-CAM++
        cam_pp, _, _ = compute_gradcam_plus_plus(model, img)
        img_uint8 = (img * 255).astype(np.uint8)
        if img_uint8.ndim == 2:
            img_uint8 = cv2.cvtColor(img_uint8, cv2.COLOR_GRAY2BGR)
        cam_pp_color = cv2.applyColorMap((cam_pp * 255).astype(np.uint8), cv2.COLORMAP_JET)
        overlay_pp = cv2.addWeighted(img_uint8, 0.5, cam_pp_color, 0.5, 0)
        overlay_pp = cv2.cvtColor(overlay_pp, cv2.COLOR_BGR2RGB)

        pred_name = CONFIG['class_display'][pred_cls]
        correct = pred_cls == y_test[idx]
        row_color = '#2ecc71' if correct else '#e74c3c'

        axes[row, 0].imshow(img, cmap='gray')
        axes[row, 0].set_ylabel(f'True: {true_cls}\nPred: {pred_name}',
                                 color=row_color, fontsize=9, fontweight='bold')

        axes[row, 1].imshow(cam_std, cmap='jet', vmin=0, vmax=1)
        axes[row, 2].imshow(overlay_std)
        axes[row, 3].imshow(cam_pp, cmap='jet', vmin=0, vmax=1)
        axes[row, 4].imshow(overlay_pp)

        for ax in axes[row]:
            ax.axis('off')

    plt.tight_layout()
    plt.savefig('gradcam_comparison.png', dpi=120, bbox_inches='tight', facecolor='#0a0a0f')
    plt.show()
    print('💾 Saved: gradcam_comparison.png')

compare_gradcam_variants(best_model, X_test, y_test)

## 📊 Section 18: Final Benchmark Report & Clinical Readiness Assessment

### Clinical Readiness Framework

For a medical AI model to be considered for clinical assistance (NOT replacement of radiologists), it must meet:

| Criterion | Threshold | Why |
|-----------|-----------|-----|
| **Sensitivity (Tumor classes)** | > 90% | Missing a tumor is life-threatening |
| **Specificity (No Tumor)** | > 85% | Limit unnecessary biopsies |
| **AUC per class** | > 0.90 | Strong discrimination ability |
| **Calibration ECE** | < 0.05 | Probabilities must be reliable |
| **Uncertainty flagging** | Configurable | High-uncertainty cases → radiologist review |

### What This Model Can Realistically Do
- ✅ Second-opinion screening tool
- ✅ Triage assistance (prioritize high-confidence cases)
- ✅ Research and education
- ❌ NOT a replacement for board-certified radiologist
- ❌ NOT validated for clinical deployment without regulatory approval (FDA 510k / CE mark)

In [ ]:
def generate_final_report(eval_results, ensemble_acc, best_model_name, best_temperature):
    """
    Generate a comprehensive clinical readiness benchmark report.
    """
    print('=' * 70)
    print('🧠 BRAIN TUMOR MRI CLASSIFICATION — FINAL BENCHMARK REPORT')
    print('=' * 70)
    print(f'  Generated: {__import__("datetime").datetime.now().strftime("%Y-%m-%d %H:%M")}')
    print(f'  Dataset:   Brain Tumor MRI (Kaggle) — 4 classes')
    print(f'  Task:      Multi-class classification (Glioma / Meningioma / No Tumor / Pituitary)')
    print()

    print('📊 MODEL BENCHMARK:')
    print(f'  {"Model":<25} {"Acc":>8} {"AUC":>8} {"F1":>8} {"Status"}')
    print('  ' + '-' * 60)
    for name, res in eval_results.items():
        acc, auc_val, f1 = res['accuracy'], res['auc_macro'], res['f1_macro']
        status = '✅ Good' if auc_val >= 0.90 else '⚠️ Marginal' if auc_val >= 0.80 else '❌ Needs work'
        star = ' ⭐' if name == best_model_name else ''
        print(f'  {name+star:<25} {acc:>8.4f} {auc_val:>8.4f} {f1:>8.4f}  {status}')

    print(f'  {"Ensemble (Weighted)":<25} {ensemble_acc:>8.4f} {"N/A":>8} {"N/A":>8}')
    print()

    print('🔬 BEST MODEL DETAIL:')
    best = eval_results[best_model_name]
    print(f'  Model: {best_model_name}')
    report = best['classification_report']
    print(f'  {"Class":<15} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
    print('  ' + '-' * 55)
    for cls in CONFIG['class_display']:
        if cls in report:
            r = report[cls]
            sensitivity_ok = '✅' if r['recall'] >= 0.85 else '⚠️'
            print(f'  {cls:<15} {r["precision"]:>10.4f} {r["recall"]:>10.4f} {r["f1-score"]:>10.4f} {int(r["support"]):>9}  {sensitivity_ok}')

    print()
    print('⚙️  DEPLOYMENT PARAMETERS:')
    print(f'  Optimal Temperature: T = {best_temperature:.3f}')
    print(f'  Uncertainty Threshold: entropy > 0.30 → flag for review')
    print(f'  Minimum Confidence: < 0.60 → flag for review')
    print()

    print('⚕️  CLINICAL READINESS ASSESSMENT:')
    criteria = [
        ('Multi-class accuracy > 80%', best['accuracy'] >= 0.80, f"{best['accuracy']:.1%}"),
        ('Macro AUC > 0.90',           best['auc_macro'] >= 0.90, f"{best['auc_macro']:.4f}"),
        ('Macro F1 > 0.80',            best['f1_macro'] >= 0.80, f"{best['f1_macro']:.4f}"),
        ('Ensemble improves accuracy', ensemble_acc >= best['accuracy'], f"{ensemble_acc:.1%}"),
        ('Calibration applied',        True, f"T={best_temperature:.2f}"),
        ('Grad-CAM explainability',    True, 'Implemented'),
        ('Uncertainty estimation',     True, 'MC Dropout (50 samples)'),
        ('Error analysis done',        True, 'Completed'),
    ]
    for criterion, met, value in criteria:
        icon = '✅' if met else '⚠️'
        print(f'  {icon} {criterion:<40} [{value}]')

    print()
    print('📌 LIMITATIONS & NEXT STEPS:')
    limitations = [
        'Trained on single dataset — needs multi-center validation',
        'Synthetic data used for demo — real Kaggle data required for deployment',
        'Requires FDA 510(k) or CE mark for clinical use',
        'Should be tested on out-of-distribution MRI scanners',
        'Consider DICOM input pipeline for production (not JPEG)',
        'Regulatory-grade logging and audit trail needed',
    ]
    for lim in limitations:
        print(f'  → {lim}')

    print()
    print('⚠️  DISCLAIMER: Research tool only. Not validated for clinical use.')
    print('=' * 70)

generate_final_report(eval_results, ensemble_acc, best_model_name, best_temperature)

## 🖥️ Section 19: Streamlit App Launch Instructions

The companion **Streamlit web application** provides a fully interactive interface for:

| Feature | Description |
|---------|-------------|
| 📤 **Image Upload** | Upload any brain MRI image (JPG/PNG) |
| 🔮 **Live Prediction** | Real-time 4-class classification |
| 🔥 **Grad-CAM Heatmap** | Interactive overlay with alpha slider |
| 📊 **Probability Chart** | Animated confidence bar chart |
| 🎲 **Uncertainty Score** | MC Dropout confidence with clinical flag |
| 🏆 **Model Selector** | Switch between all 4 trained architectures |
| 📋 **DICOM Viewer** | Basic DICOM file support |
| 📈 **Benchmark Tab** | Full model comparison dashboard |

### Launch Command
```bash
streamlit run app.py
```
Then open: http://localhost:8501

In [ ]:
# Quick launch helper
import subprocess
import sys

def launch_streamlit():
    """Launch the Streamlit app programmatically from notebook."""
    try:
        result = subprocess.Popen(
            [sys.executable, '-m', 'streamlit', 'run', 'app.py',
             '--server.headless', 'true', '--server.port', '8501'],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        print('🚀 Streamlit app launched at: http://localhost:8501')
        print('   Press Ctrl+C or stop the cell to terminate.')
        return result
    except Exception as e:
        print(f'Launch manually with: streamlit run app.py')
        print(f'Error: {e}')

print('📋 Project Summary:')
print('  📓 Notebook:   brain_tumor_classification.ipynb  (19 sections)')
print('  🖥️  App:        app.py                           (Streamlit)')
print('  📦 Models:     models/                           (saved .h5 files)')
print('  📊 Plots:      *.png                             (all visualizations)')
print()
print('🚀 To launch the web app:')
print('   streamlit run app.py')
print()
print('🔬 Models trained:')
for name, res in eval_results.items():
    print(f'   • {name:<20} Acc={res["accuracy"]:.3f}  AUC={res["auc_macro"]:.3f}')
print()
print('⚠️  Research use only. Not for clinical diagnosis.')